# RAG retrieval-quality check (Phase 1 de-risking spike)

Runs known employee questions (the top D360 article needs from the ticket audit) through the live retrieval path and prints top-k chunks per question for a human judgment call.

Run with the repo venv from the `sync-job` directory. Requires `.env` + ADC and a populated index.

Findings written up in `docs/spikes/rag-retrieval-check.md`.

In [ ]:
from molli_shared.config import get_settings
from sync_job.embedding import Embedder
from sync_job.index_store import VectorIndex

_DEPLOYED_INDEX_ID = "molli_knowledge_stream"
_PUBLIC_ENDPOINT_DOMAIN = "163164439.us-central1-719635778769.vdb.vertexai.goog"

settings = get_settings()
embedder = Embedder(settings.gcp_project_id, settings.gcp_region)
index = VectorIndex(
    project_id=settings.gcp_project_id,
    index_id=settings.vector_index_id,
    index_endpoint_id=settings.vector_index_endpoint,
    deployed_index_id=_DEPLOYED_INDEX_ID,
    public_endpoint_domain=_PUBLIC_ENDPOINT_DOMAIN,
    region=settings.gcp_region,
)

In [ ]:
# (question, what a good hit looks like) -- from the ticket audit top article needs
QUESTIONS = [
    ("How do I reset my Google password?",
     "Google password reset / account recovery (IT, highest-impact)"),
    ("How do I connect to the office printer?",
     "Connecting to office printers (IT)"),
    ("How do I request access in Entrata?",
     "Entrata: requesting access (Ops)"),
    ("How do I process a refund or reversal in Entrata?",
     "Entrata: refunds, reversals, scheduled charges (Ops)"),
    ("A resident cannot log into the resident portal, what do I do?",
     "Resident portal troubleshooting (Ops)"),
]
NEIGHBORS = 5


def show(question, expected):
    print("=" * 80)
    print(f"Q: {question}")
    print(f"   expecting: {expected}")
    print("-" * 80)
    vector = embedder.embed_query(question)
    results = index.query(vector, neighbor_count=NEIGHBORS)
    if not results:
        print("   NO RESULTS")
        return
    for i, r in enumerate(results, 1):
        section = f"  [section: {r['heading']}]" if r["heading"] else ""
        print(f"{i}. {r['title']}{section}  (distance={r['distance']:.4f})")
        print(f"   {r['url']}")
        print(f"   id={r['id']}")


for q, e in QUESTIONS:
    show(q, e)
    print()

## Findings (2026-06-12)

Full corpus (~934 articles / ~2,500 chunks). Tally: **2 hit / 1 partial / 2 miss.**

| # | Question | Verdict | Note |
|---|----------|---------|------|
| 1 | Google password reset | MISS | Right article not in top-5; collides with 2SV / social-media / portal password docs. Likely not in D360 yet. |
| 2 | Office printer | MISS | No printer article in top-5. Likely not in D360 yet. |
| 3 | Entrata request access | PARTIAL | All Entrata-flavoured but no access-request doc. Likely not in D360 yet. |
| 4 | Entrata refund/reversal | HIT | "How to Void or Refund a Payment" at #1, clear margin. |
| 5 | Resident portal login | HIT | "Reset a Resident's Portal Password" at #1, relevant rest. |

**Key finding:** the misses are *content gaps*, not pipeline failures — they
match exactly the articles the ticket audit said need creating. Where the
article exists, retrieval finds it well. Retrieval quality is gated on content
coverage, not on the embedding/chunking/search mechanism.

**Secondary:** title-only `::0` chunks rank as weak top hits (fix: prepend
title/heading to body chunks — Phase 1 follow-up); semantic collision among
password articles (Phase 2: category filter or re-rank); metadata is clean.

**Go/no-go:** conditional GO for Phase 2. Build on this retrieval layer, but
(1) prioritise SME content creation for the missing high-volume articles, and
(2) pass top-k to the LLM for disambiguation. Do not block Phase 2 on tuning.

Full write-up: `docs/spikes/rag-retrieval-check.md`.